# PRIDE LFQ group representative report

Keep this notebook and `pride_lfq_functions.py` in the same folder. The notebook contains only inputs and calls to the functions file.


In [26]:
# CELL 1 — dataset inputs, download, and LFQ names

from d_pride_lfq_functions import analyze_and_show, download_and_show, suggest_groups_and_show

ACCESSION = "PXD006299"

DATASET = {
    "accession": ACCESSION,
    "tissue_or_cell_culture": "HeLa_cell_lysate",
    "disease_type": "Heart_valve_disease_model",
    "features_list_file": "/Users/mehman/Projects/PoC_data_processing/Orthology/commons/hyperglycemia_features.txt",
    "report_file": f"{ACCESSION}_representative_report.xlsx",
    "download_dir": "PRIDE_downloads",
    "force_redownload": False,
    "timeout": 300,
    "max_full_zip_download_gb": 5.0,
}
downloaded_tables, lfq_catalog = download_and_show(DATASET)


Found 1 LFQ-containing proteinGroups table(s).


,table_id,lfq_count,gene_sources,local_file
0,table_1,6,gene column: Gene names + FASTA GN= + UniProt ...,PRIDE_downloads/PXD006299/proteinGroups_files/...


LFQ columns (6 total):


,table_id,LFQ_column_name
0,table_1,LFQ intensity Hep2_R1
1,table_1,LFQ intensity Hep2_R2
2,table_1,LFQ intensity Hep2_R3
3,table_1,LFQ intensity Jurkat_R1
4,table_1,LFQ intensity Jurkat_R2
5,table_1,LFQ intensity Jurkat_R3


In [27]:
# CELL 2 — suggest LFQ groups and show the result

GROUPS, GROUP_TABLES, grouping_preview = suggest_groups_and_show(lfq_catalog)


,suggested_group,n_LFQ_columns,detection_rule,samples
0,Hep2_R1,1,singleton / review,Hep2_R1
1,Hep2_R2,1,singleton / review,Hep2_R2
2,Hep2_R3,1,singleton / review,Hep2_R3
3,Jurkat_R1,1,singleton / review,Jurkat_R1
4,Jurkat_R2,1,singleton / review,Jurkat_R2
5,Jurkat_R3,1,singleton / review,Jurkat_R3


Suggested 6 group(s) from 6 LFQ columns.


In [28]:
# CELL 3 — review groups and analysis settings

# Optional manual corrections after reviewing the Cell 2 preview:
# GROUPS["better_name"] = GROUPS.pop("old_suggested_name")
# GROUP_TABLES["better_name"] = GROUP_TABLES.pop("old_suggested_name")
# del GROUPS["group_to_exclude"]
# GROUP_TABLES.pop("group_to_exclude", None)
MIN_GROUP_SIZE = 3

GROUPS = {
    group: columns
    for group, columns in GROUPS.items()
    if len(columns) >= MIN_GROUP_SIZE
}

GROUP_TABLES = {
    group: table
    for group, table in GROUP_TABLES.items()
    if group in GROUPS
}

grouping_preview = grouping_preview[
    grouping_preview["n_LFQ_columns"] >= MIN_GROUP_SIZE
].reset_index(drop=True)

display(grouping_preview)

SETTINGS = {
    "group_tables": GROUP_TABLES,
    "presence_threshold": 0.10,          # strict Presence > 0.10, using all LFQ columns
    "group_aggregation": "mean",
    "duplicate_gene_aggregation": "mean",
    "treat_zero_as_missing": True,
    "allow_column_reuse": False,
}


,suggested_group,n_LFQ_columns,detection_rule,samples


In [23]:
# CELL 4 — run and show the results

result = analyze_and_show(DATASET, GROUPS, SETTINGS, downloaded_tables)


Report saved to: /Users/mehman/Projects/PoC_data_processing/PXD062798_representative_report.xlsx

Gene matching and whole-table presence:


,table_id,target_genes,annotation_matched,annotation_missing,presence_rule,whole_table_LFQ_columns,eligible_after_presence,matched_but_not_eligible
0,table_1,1173,1084,89,Presence > 0.10,6,1084,0



Group representatives:


,group,table_id,LFQ_columns,aggregation,annotation_matched,eligible_after_presence,group_values_written,output_column
0,CTRL,table_1,3,mean,1084,1084,1084,PXD062798_CTRL_heart_valve_tissue_Heart_valve_...
1,RHD,table_1,3,mean,1084,1084,1084,PXD062798_RHD_heart_valve_tissue_Heart_valve_d...



Report preview:


,Gene,PXD062798_CTRL_heart_valve_tissue_Heart_valve_disease_model,PXD062798_RHD_heart_valve_tissue_Heart_valve_disease_model
0,AAAS,527203333.333333,522673333.333333
1,AARS2,56656000.0,57127000.0
2,ABCB6,44277000.0,42801333.333333
3,ABCB7,56908000.0,58260000.0
4,ABCC1,429723333.333333,404773333.333333
5,ABCC9,51334333.333333,44188666.666667
6,ABCF1,301546666.666667,395500000.0
7,ABHD16A,43685900.0,43748466.666667
8,ACAA1,163326666.666667,166473333.333333
9,ACAA2,552300000.0,487320000.0


## Output behavior

- The first report column is `Gene`, in the same order as `features_list`.
- Representative columns use `accession_group_tissue_or_cell_culture_disease`.
- Genes that match annotations but do not satisfy strict whole-table `Presence > 0.10` remain blank in the representative columns.
- Rerunning with another accession or new groups retains earlier report columns.
- If the feature list changes, use a different report filename.
- `result["feature_status"]` contains the gene-level annotation and eligibility flags when detailed checking is needed.
